# ゼロから作るショアのアルゴリズム 第4回: べき剰余

[第3回](ja/403_shor_scratch_controlled_multiplier.ipynb)で制御剰余乗算 $U_a$ を作りました。今回はそれを積み重ねて、**べき剰余** $a^j \bmod N$ を作ります。論文 III.D、式(12)(13)(14)です。

$$
\ket{j}\ket{1} \longrightarrow \ket{j}\ket{a^j \bmod N}
$$

$a^j$ は $j$ を2進数で見れば $a^{j_0} \cdot (a^2)^{j_1} \cdot (a^4)^{j_2} \cdots$ なので、**$j$ の各ビットで制御した掛け算を並べるだけ**です。前回の $U_a$ がそのまま部品になります。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Section III.D

## 今回の山場: その場で更新する

素朴に作ると困ったことになります。$x \to ax \to a^2x \to \cdots$ と掛けるたびに**新しいレジスタが要る**からです。$n$ 回掛ければレジスタが $n$ 本。これでは量子ビット数が $n^2$ に膨らみます。

論文はレジスタを増やしません。**掛けて、交換して、$a^{-1}$ で掛け戻す**という3段で、同じレジスタの中身を更新します。

1. 結果レジスタ $b$ に $ax \bmod N$ を作る（前回の乗算器）
2. $x$ と $b$ を**交換**する — これで $x$ 側が $ax$ になる
3. $a^{-1}$ を掛ける乗算器を逆向きに走らせて、$b$ を $\ket 0$ に戻す

$a$ と $N$ が互いに素なら $a^{-1} \bmod N$ が必ず存在するので、3段目が必ず効きます。

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## 交換は「無条件」でよい

ここが前回の伏線の回収です。

前回の式(11)で、$c=0$ のとき結果レジスタに $\ket 0$ ではなく $\ket x$ をコピーしました。あれがあるので、**交換に制御を付ける必要がありません**。

- $c=1$ のとき: $\ket{x,\ ax}$ を交換して $\ket{ax,\ x}$
- $c=0$ のとき: $\ket{x,\ x}$ を交換しても $\ket{x,\ x}$ のまま

$c=0$ で何も起きないのは、**交換を止めたからではなく、交換しても同じ状態だから**です。もし前回コピーを省いて $\ket{x, 0}$ にしていたら、交換で $\ket{0, x}$ になり $x$ が失われます。論文が式(11)にコピーを入れているのは、この一点のためです。

交換は CX 3つで書けます。

In [2]:
from blueqat import Circuit


def carry(c, a, b, c_next):
    q = Circuit(); q.ccx[a, b, c_next]; q.cx[a, b]; q.ccx[c, b, c_next]; return q

def carry_inv(c, a, b, c_next):
    q = Circuit(); q.ccx[c, b, c_next]; q.cx[a, b]; q.ccx[a, b, c_next]; return q

def sum_gate(c, a, b):
    q = Circuit(); q.cx[a, b]; q.cx[c, b]; return q

def add_circuit(c, a, b, n):
    q = Circuit()
    for i in range(n - 1):
        q += carry(c[i], a[i], b[i], c[i + 1])
    q += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    q.cx[a[n - 1], b[n - 1]]
    q += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        q += carry_inv(c[i], a[i], b[i], c[i + 1])
        q += sum_gate(c[i], a[i], b[i])
    return q

def sub_circuit(c, a, b, n):
    r = Circuit(); r.ops = list(reversed(add_circuit(c, a, b, n).ops)); return r

def set_to_zero_if(t, N, val_N, n):
    """t == 0 のとき N レジスタを 0 にする。自分自身が逆演算。"""
    q = Circuit(); q.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            q.cx[t, N[j]]
    q.x[t]; return q

def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N。論文 Fig.4。"""
    q = Circuit()
    q += add_circuit(c, a, b, n)
    q += sub_circuit(c, N, b, n)
    q.cx[b[n], t]
    g = set_to_zero_if(t, N, val_N, n)
    q += g; q += add_circuit(c, N, b, n); q += g
    q += sub_circuit(c, a, b, n)
    q.x[b[n]]; q.cx[b[n], t]; q.x[b[n]]
    q += add_circuit(c, a, b, n)
    return q

In [3]:
def swap_circuit(p, q_reg):
    """⭐ 無条件スワップ。CX 3つで1組を交換する。"""
    s = Circuit()
    for u, v in zip(p, q_reg):
        s.cx[u, v]; s.cx[v, u]; s.cx[u, v]
    return s

## 制御乗算 $U_a$ ― その場で更新する形

前回の乗算器を、3段（掛ける・交換する・掛け戻す）に組み直します。加算器は毎段同じものなので、ループの外で一度だけ組んで使い回します。

In [4]:
def layout(n):
    c     = list(range(n))
    const = list(range(n, 2 * n))
    b     = list(range(2 * n, 3 * n + 1))
    N     = list(range(3 * n + 1, 4 * n + 1))
    x     = list(range(4 * n + 1, 5 * n + 1))
    return c, const, b, N, x

def build_Ua(mult_a, val_N, n, c, const, b, N, x, ctrl, t, adder, adder_inv):
    """その場で更新する制御乗算 U_a：掛けて・交換して・a^-1 で掛け戻す。"""
    circ = Circuit()
    # A: b += (a*x) mod N ── Fig.5。制御するのは定数のロードだけ
    for i in range(n):
        cv = (mult_a * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (cv >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]
        circ += load; circ += adder; circ += load
    # 式(11)：ctrl == 0 なら x を b へコピー
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]
    # B: ⭐ 無条件スワップ（ctrl=0 でも |x,x> なので交換しても変わらない）
    circ += swap_circuit(x, b[:n])
    # C: b -= (a^-1 * x) mod N ── b を |0> に戻す
    a_inv = pow(mult_a, -1, val_N)
    for i in range(n):
        cv = (a_inv * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (cv >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]
        circ += load; circ += adder_inv; circ += load
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]
    return circ

In [5]:
def run_Ua(mult_a, val_x, val_N, ctrl_value, n=None, quiet=False):
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    c, const, b, N, x = layout(n)
    ctrl, t = 5 * n + 1, 5 * n + 2
    circ = Circuit(5 * n + 3)
    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1: circ.x[x[i]]
        if (val_N >> i) & 1: circ.x[N[i]]
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)
    adder_inv = Circuit(); adder_inv.ops = list(reversed(adder.ops))
    circ += build_Ua(mult_a, val_N, n, c, const, b, N, x, ctrl, t, adder, adder_inv)
    total = circ.n_qubits
    if not quiet:
        print("  [n=%d, qubits=%d, gates=%d]" % (n, total, len(circ.ops)))
    st = circ.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    rd = lambda reg: int("".join(reversed([st[total - 1 - q] for q in reg])), 2)
    return {"x": rd(x), "b": rd(b), "N": rd(N), "const": rd(const),
            "c": rd(c), "t": int(st[total - 1 - t]), "qubits": total,
            "gates": len(circ.ops)}

$c=0$ のとき $x$ がそのまま残り、$c=1$ のときだけ掛け算が起きること、そして**作業用のレジスタがすべて $\ket 0$ に戻っていること**を確認します。戻っていないと、後段で重ね合わせの干渉が壊れます。

In [6]:
for a, x, ctrl in [(2, 1, 1), (2, 2, 1), (2, 1, 0), (2, 2, 0), (2, 0, 1)]:
    r = run_Ua(a, x, 3, ctrl, 2, quiet=True)
    expected = (a * x) % 3 if ctrl else x
    clean = (r["b"] == 0 and r["N"] == 3 and r["const"] == 0
             and r["c"] == 0 and r["t"] == 0)
    ok = r["x"] == expected and clean
    print(f"ctrl={ctrl}: {a}*{x} mod 3 -> x={r['x']} (expected {expected}), "
          f"scratch clean={clean}  {'OK' if ok else 'MISMATCH'}")
print(f"[{r['qubits']} qubits, {r['gates']} gates]")

ctrl=1: 2*1 mod 3 -> x=2 (expected 2), scratch clean=True  OK


ctrl=1: 2*2 mod 3 -> x=1 (expected 1), scratch clean=True  OK


ctrl=0: 2*1 mod 3 -> x=1 (expected 1), scratch clean=True  OK


ctrl=0: 2*2 mod 3 -> x=2 (expected 2), scratch clean=True  OK


ctrl=1: 2*0 mod 3 -> x=0 (expected 0), scratch clean=True  OK
[13 qubits, 354 gates]


## べき剰余

$U_a$ が部品になったので、あとは指数レジスタの各ビット $j_i$ で $U_{a^{2^i}}$ を制御するだけです。$a^{2^i} \bmod N$ は古典で先に計算しておきます（論文の式(13)）。

In [7]:
def build_modexp(mult_a, val_N, exp_val, n, n_exp):
    """|j>|1> → |j>|a^j mod N>。j は古典の値として X で置く。"""
    c, const, b, N, x = layout(n)
    t   = 5 * n + 2
    exp = list(range(5 * n + 3, 5 * n + 3 + n_exp))   # 指数レジスタ
    circ = Circuit(5 * n + 3 + n_exp)
    for i in range(n_exp):
        if (exp_val >> i) & 1:
            circ.x[exp[i]]
    circ.x[x[0]]                                      # x := 1 から始める
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)
    adder_inv = Circuit(); adder_inv.ops = list(reversed(adder.ops))
    for i in range(n_exp):
        a_i = pow(mult_a, 1 << i, val_N)              # a^(2^i) mod N
        circ += build_Ua(a_i, val_N, n, c, const, b, N, x, exp[i], t,
                         adder, adder_inv)
    return circ, x, b, N, c, const, t

def run_modexp(mult_a, val_N, exp_val, n=None, n_exp=None, quiet=False):
    if n is None:
        n = val_N.bit_length()
    if n_exp is None:
        n_exp = max(1, exp_val.bit_length())
    circ, x, b, N, c, const, t = build_modexp(mult_a, val_N, exp_val, n, n_exp)
    total = circ.n_qubits
    if not quiet:
        print("  [n=%d, exp bits=%d, qubits=%d, gates=%d]"
              % (n, n_exp, total, len(circ.ops)))
    st = circ.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    rd = lambda reg: int("".join(reversed([st[total - 1 - q] for q in reg])), 2)
    return {"x": rd(x), "b": rd(b), "N": rd(N), "const": rd(const),
            "c": rd(c), "t": int(st[total - 1 - t]),
            "qubits": total, "gates": len(circ.ops)}

In [8]:
for j in range(4):
    r = run_modexp(2, 3, j, 2, 2, quiet=True)
    expected = pow(2, j, 3)
    clean = r["b"] == 0 and r["const"] == 0 and r["c"] == 0 and r["t"] == 0
    print(f"2^{j} mod 3 = {r['x']}  (Python: {expected}, "
          f"scratch clean={clean})  {'OK' if r['x'] == expected and clean else 'MISMATCH'}")
print(f"[{r['qubits']} qubits, {r['gates']} gates]")

2^0 mod 3 = 1  (Python: 1, scratch clean=True)  OK


2^1 mod 3 = 2  (Python: 2, scratch clean=True)  OK


2^2 mod 3 = 1  (Python: 1, scratch clean=True)  OK


2^3 mod 3 = 2  (Python: 2, scratch clean=True)  OK
[15 qubits, 706 gates]


## 網羅テスト

$N=3$ と $N=5$ について、$N$ と互いに素な $a$ すべて × $x$ すべて × $c$ の両方を試します。値が合っているかだけでなく、毎回**作業レジスタが $\ket 0$ に戻っているか**も見ます。

実測（statevector）:

| | 量子ビット | ケース数 | 結果 | 時間 |
|---|---|---|---|---|
| $N=3$ ($n=2$) | 13 | 12 | 全通過 | 15秒 |
| $N=5$ ($n=3$) | 18 | 40 | 全通過 | 1293秒 |

$N=7$ 以上は状態ベクトルが重く、網羅は現実的ではありません。$N=3$ の1ケースが約1.3秒に対し、$N=5$ は約32秒。量子ビットが5本増えて振幅が32倍、実測も約24倍です。

In [9]:
from math import gcd

for N in (3,):                      # N=5 は 1293 秒かかるのでここでは N=3 だけ
    n = N.bit_length()
    ok = ng = 0
    for a in range(1, N):
        if gcd(a, N) != 1:
            continue
        for x in range(N):
            for ctrl in (0, 1):
                r = run_Ua(a, x, N, ctrl, n, quiet=True)
                expected = (a * x) % N if ctrl else x
                good = (r["x"] == expected and r["b"] == 0 and r["N"] == N
                        and r["const"] == 0 and r["c"] == 0 and r["t"] == 0)
                ok += good
                ng += not good
    print(f"N={N} (n={n}): PASS {ok} / FAIL {ng}")

N=3 (n=2): PASS 12 / FAIL 0


## 規模

前回の乗算器が $5n+3$ 量子ビットでした。べき剰余は指数レジスタを足すだけなので $5n+3+n_{exp}$ です。レジスタを増やさずにその場で更新する、という設計がそのまま効いています。

実測（$a=2$, $N=3$, $n=2$, 指数2ビット）: **15量子ビット / 706ゲート**。

論文 IV 節はべき剰余を $O(n^3)$ の基本ゲートと見積もっています。こちらは $U_a$ を $n_{exp}$ 回並べ、$U_a$ の中で剰余加算器を $2n$ 回使い、剰余加算器が $O(n)$ ゲートなので、同じ $O(n^3)$ です。

一方、古典シミュレーションのコストは量子ビット数に対して**指数**です。$N=3$ で13量子ビット・1.3秒、$N=5$ で18量子ビット・32秒。この差は回路が大きくなったからではなく、状態ベクトルが $2^{13}$ から $2^{18}$ に増えたからです。

## まとめ

- べき剰余は、**指数の各ビットで制御した掛け算を並べるだけ**（式(12)(13)）
- レジスタを増やさないために、**掛けて・交換して・$a^{-1}$ で掛け戻す**の3段でその場更新する
- **交換に制御は要らない。** 前回の式(11)で $c=0$ のとき $\ket x$ をコピーしておいたので、交換しても状態が変わらない。コピーはこのための伏線だった
- $N=3$ で12ケース、$N=5$ で40ケース、値と作業ビットの復帰まで全通過
- 15量子ビット / 706ゲート（$N=3$, 指数2ビット）

## 次回予告

ここまでで、$j$ を**古典の値**として与えれば $a^j \bmod N$ が計算できるようになりました。

最終回では、指数レジスタに $X$ ではなく $H$ を置きます。**それだけ**で、すべての $j$ について同時に $a^j \bmod N$ が計算されます。回路は一文字も変えません。そこへ逆量子フーリエ変換をかけて位数を取り出し、第1回の古典後処理につなげば、ショアのアルゴリズムが最初から最後までつながります。